In [ ]:
import os
import pandas as pd
import numpy as np
from geopy.distance import geodesic

# Define paths
input_directory = '/mnt/scratch_lustre/barthelx/Masrur/Nowcasting/Input_Data/Sydney_Data'
station_info_path = '/mnt/scratch_lustre/barthelx/Masrur/Nowcasting/Input_Data/Selected_Station_info.csv'
target_station = 'LIDCOMBE'  # Target station
output_file = f'/mnt/scratch_lustre/barthelx/Masrur/Nowcasting/Input_Data/{target_station}_Influence_Enhanced.csv'

# Load station information (with geographic coordinates)
station_df = pd.read_csv(station_info_path)
station_coords = station_df.set_index('SiteName')[['Latitude', 'Longitude']]

# Get target station's coordinates
target_coords = tuple(station_coords.loc[target_station])

# Initialize a DataFrame for the target station
target_data = None

# Variables of interest
variables = ["H2O", "HUMID", "NO", "NOX", "PM10", "PM2.5", "RAIN", "SO2", "TEMP", "WDR", "WGU", "WSP"]
variable_influences = {var: [] for var in variables}


# Function to fix "24:00" issue in time and generate valid timestamps
def handle_time_errors(row):
    date, time = row['Date'], row['Time']
    if time == '24:00':
        # Convert "24:00" to "00:00" and increment the date by one day
        new_date = pd.to_datetime(date, format='%d/%m/%Y') + pd.Timedelta(days=1)
        return new_date.strftime('%d/%m/%Y') + ' 00:00'
    else:
        return date + ' ' + time

# Iterate over all CSV files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith('.csv'):
        # Identify the study site from the filename (first word separated by underscore)
        study_site = filename.split('_')[0]

        # Load the CSV file
        file_path = os.path.join(input_directory, filename)
        df = pd.read_csv(file_path)

        # Ensure 'Date' and 'Time' columns exist
        if 'Date' not in df.columns or 'Time' not in df.columns:
            print(f"Skipping {filename}: Missing Date/Time columns.")
            continue

        # Apply the fix to handle "24:00" times
        df['Timestamp'] = df.apply(handle_time_errors, axis=1)
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%d/%m/%Y %H:%M')

        # Process target station data
        if study_site == target_station:
            target_data = df[['Timestamp'] + variables]
        else:
            # Get coordinates of the current station
            study_coords = tuple(station_coords.loc[study_site])

            # Calculate distance to the target station
            distance = geodesic(target_coords, study_coords).kilometers
            weight = 1 / distance if distance > 0 else 1  # Inverse distance weight

            # Compute weighted variables and store
            for var in variables:
                if var in df.columns:
                    weighted_var = df[var] * weight
                    variable_influences[var].append(weighted_var)

# Aggregate influences for each variable
influence_df = pd.DataFrame({'Timestamp': target_data['Timestamp']})
for var, weighted_data in variable_influences.items():
    if weighted_data:
        # Sum weighted data across all stations
        influence_df[f"{var}_Influence"] = pd.concat(weighted_data, axis=1).sum(axis=1)
    else:
        # No data for this variable, fill with NaN
        influence_df[f"{var}_Influence"] = np.nan

# Combine with the target station data
final_data = pd.merge(target_data, influence_df, on='Timestamp', how='left')

# Save the augmented dataset
final_data.to_csv(output_file, index=False)
print(f"Processed dataset saved to {output_file}.")


Processed dataset saved to /mnt/scratch_lustre/barthelx/Masrur/Nowcasting/Input_Data/LIDCOMBE_Influence_Enhanced.csv.
